In [2]:
import pyspark
from pyspark.sql import SparkSession, types, functions as F
import pandas as pd
from datetime import datetime
import requests
import os
from concurrent.futures import ThreadPoolExecutor

spark = SparkSession.builder.master("local[*]").appName("test").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/17 09:49:25 WARN Utils: Your hostname, GUI-NOTEBOOK, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/17 09:49:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 09:49:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
CHUNK_SIZE = 10 * 1024 * 1024

In [4]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True), 
    types.StructField('dispatching_base_num', types.StringType(), True), 
    types.StructField('originating_base_num', types.StringType(), True), 
    types.StructField('request_datetime', types.TimestampType(), True), 
    types.StructField('on_scene_datetime', types.TimestampType(), True), 
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.LongType(), True), 
    types.StructField('DOLocationID', types.LongType(), True), 
    types.StructField('trip_miles', types.DoubleType(), True), 
    types.StructField('trip_time', types.LongType(), True), 
    types.StructField('base_passenger_fare', types.DoubleType(), True), 
    types.StructField('tolls', types.DoubleType(), True), 
    types.StructField('bcf', types.DoubleType(), True), 
    types.StructField('sales_tax', types.DoubleType(), True), 
    types.StructField('congestion_surcharge', types.DoubleType(), True), 
    types.StructField('airport_fee', types.DoubleType(), True), 
    types.StructField('tips', types.DoubleType(), True), 
    types.StructField('driver_pay', types.DoubleType(), True), 
    types.StructField('shared_request_flag', types.StringType(), True), 
    types.StructField('shared_match_flag', types.StringType(), True), 
    types.StructField('access_a_ride_flag', types.StringType(), True), 
    types.StructField('wav_request_flag', types.StringType(), True), 
    types.StructField('wav_match_flag', types.StringType(), True)
    ])

In [5]:
def get_taxi_data(taxi_type, start_date, end_date):
    base_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data'

    start_date =  datetime.strptime(start_date, "%Y-%m").date()
    end_date =  datetime.strptime(end_date, "%Y-%m").date()
    os.makedirs(f'data/{taxi_type}', exist_ok=True)

    months = []
    current = start_date.replace(day=1)

    while current <= end_date:
        months.append(current)
        if current.month == 12:
            current = current.replace(year=current.year +1, month=1)
        else:
            current = current.replace(month=current.month + 1)

    def download_single_file(file_path, url):
        print(f'Starting: {os.path.basename(url)}')
        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()
            with open(file_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                    f.write(chunk)
        except requests.exceptions.RequestException as e:
            print(f'File not found or not available: {e}')
    with ThreadPoolExecutor(max_workers=6) as executor:
        for month_date in months:
            download_url = f"{base_url}/{taxi_type}_tripdata_{month_date.strftime('%Y-%m')}.parquet"
            file_path = f"data/{taxi_type}/{os.path.basename(download_url)}"
            executor.submit(download_single_file, file_path, download_url)
        
        

In [50]:
taxi_type = 'fhvhv'
start_date = '2025-01'
end_date = '2025-12'

get_taxi_data(taxi_type, start_date, end_date)

Starting: fhvhv_tripdata_2025-01.parquet
Starting: fhvhv_tripdata_2025-02.parquet
Starting: fhvhv_tripdata_2025-03.parquet
Starting: fhvhv_tripdata_2025-04.parquet
Starting: fhvhv_tripdata_2025-05.parquet
Starting: fhvhv_tripdata_2025-06.parquet
Starting: fhvhv_tripdata_2025-07.parquet
Starting: fhvhv_tripdata_2025-08.parquet
Starting: fhvhv_tripdata_2025-09.parquet
Starting: fhvhv_tripdata_2025-10.parquet
Starting: fhvhv_tripdata_2025-11.parquet
Starting: fhvhv_tripdata_2025-12.parquet


In [6]:
fhv_df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .parquet(os.path.abspath('data/fhvhv'))

In [7]:
fhv_df.show()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+-----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee| tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+

In [8]:
fhv_df.count()

243589684

In [9]:
spark.version

'4.1.1'

In [10]:
homework_taxi_type = 'yellow'
homework_start_date = '2025-01'
homework_end_date = '2025-12'

get_taxi_data(homework_taxi_type, homework_start_date, homework_end_date)

Starting: yellow_tripdata_2025-01.parquet
Starting: yellow_tripdata_2025-02.parquet
Starting: yellow_tripdata_2025-03.parquet
Starting: yellow_tripdata_2025-04.parquet
Starting: yellow_tripdata_2025-05.parquet
Starting: yellow_tripdata_2025-06.parquet
Starting: yellow_tripdata_2025-07.parquet
Starting: yellow_tripdata_2025-08.parquet
Starting: yellow_tripdata_2025-09.parquet
Starting: yellow_tripdata_2025-10.parquet
Starting: yellow_tripdata_2025-11.parquet
Starting: yellow_tripdata_2025-12.parquet


In [16]:
yellow_df = spark.read \
    .parquet(os.path.abspath('data/yellow/yellow_tripdata_2025-11.parquet'))

In [17]:
yellow_df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [18]:
yellow_df.count()

4181444

In [20]:
if not os.path.exists('homework/'):
    os.makedirs('homework/')

yellow_df.repartition(4).write.parquet('homework/', mode='overwrite')

In [29]:
yellow_df \
    .filter(F.to_date(yellow_df.tpep_pickup_datetime) == '2025-11-15') \
    .count()

162604

In [30]:
yellow_df.createOrReplaceTempView("yellow_tripdata")

In [41]:
spark.sql("""
                              SELECT
                                *,
                                (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 3600 AS trip_hours
                              FROM
                              yellow_tripdata yt
                              WHERE (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) = (SELECT MAX((UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime))) FROM yellow_tripdata)
""") \
.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|       trip_hours|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------------+
|       2| 2025-11-26 20:22:12|  2025-11-30 15:01:00|              1|       

In [40]:
yellow_df \
    .withColumn('trip_hours', (F.unix_timestamp(yellow_df.tpep_dropoff_datetime) - F.unix_timestamp(yellow_df.tpep_pickup_datetime)) / 3600) \
    .orderBy('trip_hours', ascending=False) \
    .limit(1) \
    .show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|       trip_hours|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------------+
|       2| 2025-11-26 20:22:12|  2025-11-30 15:01:00|              1|       

In [42]:
zones_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("taxi_zone_lookup.csv")

In [44]:
zones_df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [43]:
zones_df.createOrReplaceTempView("taxi_zones")

In [49]:
spark.sql("""
          SELECT
            yt.PULocationID,
            tz.Zone,
            COUNT(*) AS total_trips
          FROM yellow_tripdata yt
          INNER JOIN taxi_zones tz
            ON yt.PULocationID = tz.LocationID
          GROUP BY
            yt.PULocationID,
            tz.Zone
          ORDER BY
            total_trips
          LIMIT 10
""").show()

+------------+--------------------+-----------+
|PULocationID|                Zone|total_trips|
+------------+--------------------+-----------+
|           5|       Arden Heights|          1|
|          84|Eltingville/Annad...|          1|
|         105|Governor's Island...|          1|
|         187|       Port Richmond|          3|
|         199|       Rikers Island|          4|
|         204|   Rossville/Woodrow|          4|
|         111| Green-Wood Cemetery|          4|
|         109|         Great Kills|          4|
|           2|         Jamaica Bay|          5|
|         251|         Westerleigh|         12|
+------------+--------------------+-----------+

